In [ ]:
import matplotlib.pyplot as plt
import pickle
import numpy as np
import joblib as jl
from tqdm import tqdm
import similaritymeasures as sm

from functions_synthetic_data_generation import *
from functions_synthetic_data_analysis import *
from functions_hmm import *
import scipy as sp


plt.style.use('../paper.mplstyle')

print(plt.get_backend())
%matplotlib qt5
print(plt.get_backend())

module://matplotlib_inline.backend_inline
qt5agg


# Simulations generations

## Parameters

In [ ]:
from DDM_vXbis.used_parameters_n_50 import *

# Plot HMM fitting result

# Drift Estimation

## Generating "theoretical" average probability sequences

In [4]:
DRIFT_RANGE = np.linspace(0.001,0.11,200)

generate_theoretical_sequences = False # PARAM

if generate_theoretical_sequences:

    temp_args_dict = copy.deepcopy(DEFAULT_ARGS_DICT)
        
    args = [temp_args_dict] + [5000] + [P_CW_INIT_RANGE]
    DRIFT_RANGE = np.linspace(0.001,0.11,50)

    test_average_probability_sequences_list = generate_test_average_probability_sequences_identical_drifts_random_init(DRIFT_RANGE, args)

    with open(f'{MAIN_FOLDER_PATH}/test_average_probability_sequences.pkl', 'wb') as file:
        pickle.dump([DRIFT_RANGE, test_average_probability_sequences_list], file)

else:
   
    with open(f'{MAIN_FOLDER_PATH}/test_average_probability_sequences.pkl', 'rb') as file:
        DRIFT_RANGE, test_average_probability_sequences_list = pickle.load(file)
test_average_probability_sequences_list = [test_seq[:STEPS_NUMBER] for test_seq in test_average_probability_sequences_list]
print(len(test_average_probability_sequences_list[0]))

40


In [ ]:
np.linspace(0.001,0.11,50)

array([0.001     , 0.00322449, 0.00544898, 0.00767347, 0.00989796,
       0.01212245, 0.01434694, 0.01657143, 0.01879592, 0.02102041,
       0.0232449 , 0.02546939, 0.02769388, 0.02991837, 0.03214286,
       0.03436735, 0.03659184, 0.03881633, 0.04104082, 0.04326531,
       0.0454898 , 0.04771429, 0.04993878, 0.05216327, 0.05438776,
       0.05661224, 0.05883673, 0.06106122, 0.06328571, 0.0655102 ,
       0.06773469, 0.06995918, 0.07218367, 0.07440816, 0.07663265,
       0.07885714, 0.08108163, 0.08330612, 0.08553061, 0.0877551 ,
       0.08997959, 0.09220408, 0.09442857, 0.09665306, 0.09887755,
       0.10110204, 0.10332653, 0.10555102, 0.10777551, 0.11      ])

In [5]:
example_set = 0
simu_index = 3

stacked_proba_sequences = []
stacked_predicted_proba_sequences = []

fig = plt.figure(figsize=(7, 2.5), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(2, 1, hspace=0.3, height_ratios=(1,0.3))
ax1 = plt.subplot(row[0,0])
ax2 = plt.subplot(row[1,0])

for simu_index in range(0,50):

    with open(f'{MAIN_FOLDER_PATH}/drift_{2}/simulation_set_{example_set}.pkl', 'rb') as file:
        simulations_set = pickle.load(file)

    with open(f'{MAIN_FOLDER_PATH}/drift_{2}/hmm_fit_output_{example_set}.pkl', 'rb') as file:
        fit_output = pickle.load(file)

    model = fit_output['models'][1]
    choices_sequence = [synth_data['choices'] for synth_data in simulations_set][simu_index]
    rewards_sequence = [synth_data['rewards'] for synth_data in simulations_set][simu_index]
    proba_sequence = [synth_data['p_cw'] for synth_data in simulations_set][simu_index]

    emissionmat = model.emissionprob_
    startprob = model.startprob_

    predicted_states_sequence = model.predict(np.reshape(choices_sequence,(-1,1)))
    predicted_proba_sequence = np.array([emissionmat[s,1] for s in predicted_states_sequence])
    colors = ('indigo','violet')
    colors_sequence = [colors[r] for r in rewards_sequence]

    ###

    steps = np.arange(len(proba_sequence))


    # ax1.scatter(steps, proba_sequence, marker='+', c=colors_sequence, s=70, linewidth=1.5, alpha=0.7)
    # ax1.plot(steps, proba_sequence, color='k', alpha=0.5)

    # ax1.scatter(steps, predicted_proba_sequence, marker='+', c='green', s=70, linewidth=1.5, alpha=0.5)
    # ax1.plot(steps, predicted_proba_sequence, alpha=0.5)
    
    # for s, p in enumerate(emissionmat[:,1]):
    #     ax1.axhline(p, linestyle='--', color='grey', linewidth=0.5, zorder=0)
    #     ax1.text(-1.7, p+0.05, rf'$P_{{s_{s}}}$', fontsize=7)

    # ax1.set_yticks(np.arange(0,1.1,0.2))
    # ax1.set_ylabel(r'$P_n$')
    # ax1.set_ylim(-0.1,1.1)

    ###


    # ax2.scatter(steps, choices_sequence, marker='+', c=colors_sequence, linewidth=1.5)
    # ax2.set_yticks((0,1))
    # ax2.set_yticklabels(('CCW', 'CW'))
    # ax2.set_ylim(-0.2,1.2)
    # ax2.set_xlabel('Step')

    stacked_proba_sequences.append(proba_sequence)
    stacked_predicted_proba_sequences.append(predicted_proba_sequence)

# ax1.plot(steps, np.mean(stacked_proba_sequences,axis=0), color='k', alpha=1)
ax1.plot(steps, np.mean(stacked_predicted_proba_sequences,axis=0), color='green', alpha=1)

for idx in (50,95,150,-1):
    ax1.plot(steps, test_average_probability_sequences_list[idx], color='coral', ls='--', alpha=0.5)
ax1.set_ylim(-0.1,1.1)


(-0.1, 1.1)

## Maximizing likelihood

In [41]:
def compute_proba_distri_sequence(args_dict, p_cw_init_range, n_simulations):

    proba_distri_sequence = []

    for _ in tqdm(range(n_simulations), leave=False, desc='Simulations : '):

        temp_args_dict = copy.deepcopy(args_dict)
        temp_args_dict['p_cw_init'] = np.random.choice(p_cw_init_range)

        ddm_result = run_simulation(temp_args_dict)

        proba_distri_sequence.append(ddm_result['p_cw'])

    return proba_distri_sequence

In [63]:
def compute_distri_likelihood(data_proba_distri_sequence, proba_distri_sequence):

    steps = np.arange(len(data_proba_distri_sequence))

    interp_proba_distri_func = sp.interpolate.make_interp_spline(steps, proba_distri_sequence, k=1)

    likelihood_arr = interp_proba_distri_func(np.array(data_proba_distri_sequence))

    return likelihood_arr

def compute_choice_likelihood(choice, p_cw):

    likelihood = p_cw if choice==1 else 1-p_cw

    return likelihood

def compute_sequence_likelihood(choices_sequences_arr, proba_distri_sequence):

    choices_sequences_arr = np.array([synth_data['choices'] for synth_data in simulations_set])

    data_proba_distri_sequence = np.sum(choices_sequences_arr, axis=0)/STEPS_NUMBER

    steps = np.arange(len(data_proba_distri_sequence))

    distri_likelihood_arr = compute_distri_likelihood(data_proba_distri_sequence, proba_distri_sequence)

    for seq in choices_sequences_arr:
        for n in steps:

            likelihood = likelihood + compute_choice_likelihood(seq[n])

In [94]:
# 1 # Compute data probability distribution for every step

example_set = 0

stacked_proba_sequences = []
stacked_predicted_proba_sequences = []

with open(f'{MAIN_FOLDER_PATH}/drift_{2}/simulation_set_{example_set}.pkl', 'rb') as file:
    simulations_set = pickle.load(file)

with open(f'{MAIN_FOLDER_PATH}/drift_{2}/hmm_fit_output_{example_set}.pkl', 'rb') as file:
    fit_output = pickle.load(file)

choices_sequences_arr = np.array([synth_data['choices'] for synth_data in simulations_set])

data_proba_distri_sequence = np.sum(choices_sequences_arr, axis=0)/STEPS_NUMBER


# 2 #

drift = 0.0525

# 3 #

generate_theoretical_sequences = False # PARAM

if generate_theoretical_sequences:

    test_probability_sequences_list = []
    temp_args_dict = copy.deepcopy(DEFAULT_ARGS_DICT)
        
    DRIFT_RANGE = np.linspace(0.001,0.11,50)

    for drift in tqdm(DRIFT_RANGE, desc='Drift : '):
    
        drift_matrix = np.array([[drift, -drift],
                                 [0    , 0     ]])
        
        temp_args_dict['drift_matrix'] = drift_matrix
        
        test_probability_sequences_list.append(compute_proba_distri_sequence(temp_args_dict, P_CW_INIT_RANGE, 500))


    with open(f'{MAIN_FOLDER_PATH}/test_probability_distri_sequences.pkl', 'wb') as file:
        pickle.dump([DRIFT_RANGE, test_probability_sequences_list], file)

else:
   
    with open(f'{MAIN_FOLDER_PATH}/test_probability_distri_sequences.pkl', 'rb') as file:
        DRIFT_RANGE, test_probability_sequences_list = pickle.load(file)

# 4 # Compute likelihood

sequence_likelihood_list = []

for i, drift in enumerate(tqdm(DRIFT_RANGE, desc='Drift : ')):

    likelihood = np.prod(compute_distri_likelihood(data_proba_distri_sequence, np.transpose(test_probability_sequences_list[i])), axis=1)
    
    sequence_likelihood_list.append(np.mean(likelihood))


plt.plot(DRIFT_RANGE, np.log(sequence_likelihood_list))
# test_average_probability_sequences_list = [test_seq[:STEPS_NUMBER] for test_seq in test_average_probability_sequences_list]
# print(len(test_average_probability_sequences_list[0]))

# for seq in 

Drift : 100%|██████████| 50/50 [00:00<00:00, 1809.23it/s]


In [ ]:
def estimate_drift(simulations_set, model, test_average_probability_sequences_list, drift_range):

    choices_sequences_list = [synth_data['choices'] for synth_data in simulations_set]

    average_proba_sequence_hmm = compute_reconstructed_average_proba_sequence(choices_sequences_list, model)

    mse_list = []

    for test_average_probability_sequence in test_average_probability_sequences_list:
        
        mse_list.append(compute_mean_square_error_v2(average_proba_sequence_hmm, test_average_probability_sequence))

    min_mse = np.min(mse_list)
    min_mse_index = np.where(mse_list==min_mse)[0]
    recovered_drift = drift_range[min_mse_index]

    return recovered_drift, min_mse_index

In [29]:
def estimate_drift(simulations_set, model, test_average_probability_sequences_list, drift_range):

    choices_sequences_list = [synth_data['choices'] for synth_data in simulations_set]

    average_proba_sequence_hmm = compute_reconstructed_average_proba_sequence(choices_sequences_list, model)

    area_list = []

    steps = np.arange(len(average_proba_sequence_hmm))

    for test_average_probability_sequence in test_average_probability_sequences_list:
        
        area = sm.area_between_two_curves(np.array([steps,average_proba_sequence_hmm]),np.array([steps,test_average_probability_sequence]))

        area_list.append(area)        

    
    min_area = np.min(area_list)
    min_area_index = np.where(area_list==min_area)[0]
    recovered_drift = drift_range[min_area_index]

    return recovered_drift, min_area_index

In [27]:
recovered_drift, index = estimate_drift(simulations_set, model, test_average_probability_sequences_list, DRIFT_RANGE)
print(index)

(array([192]),)
[np.float64(0.05160989670381161), np.float64(0.05251213494409129), np.float64(0.05137081287185857), np.float64(0.05096586527608499), np.float64(0.05362233663639099), np.float64(0.051014815030181127), np.float64(0.05670999794754483), np.float64(0.05037636301031484), np.float64(0.05142324625187428), np.float64(0.05196014428943879), np.float64(0.052393810155273224), np.float64(0.053207128354495425), np.float64(0.04832310022518371), np.float64(0.049598403665473856), np.float64(0.05360613212076332), np.float64(0.04945745126365836), np.float64(0.05288549059911096), np.float64(0.04798579256809299), np.float64(0.05018117555634383), np.float64(0.05197515398649494), np.float64(0.050825001465966746), np.float64(0.05196016061540243), np.float64(0.05206170953063172), np.float64(0.05138919864093233), np.float64(0.05220687040328825), np.float64(0.05025553577379316), np.float64(0.04999704677992989), np.float64(0.04674512782852097), np.float64(0.051556164419785555), np.float64(0.0497408

In [7]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

for i in range(N_SETS):

    with open(f'{MAIN_FOLDER_PATH}/drift_{2}/simulation_set_{i}.pkl', 'rb') as file:
        simulations_set = pickle.load(file)

    with open(f'{MAIN_FOLDER_PATH}/drift_{2}/hmm_fit_output_{i}.pkl', 'rb') as file:
        fit_output = pickle.load(file)

    model = fit_output['models'][0]
    choices_sequences_list = [synth_data['choices'] for synth_data in simulations_set]

    average_proba_sequence_hmm = compute_reconstructed_average_proba_sequence(choices_sequences_list, model)

    temp_mse_list = []

    for test_average_probability_sequence in test_average_probability_sequences_list:
            
        temp_mse_list.append(compute_mean_square_error_v2(average_proba_sequence_hmm, test_average_probability_sequence))

    min_mse = np.min(temp_mse_list)
    min_mse_index = np.where(temp_mse_list==min_mse)[0]

    ax1.plot(DRIFT_RANGE, temp_mse_list, alpha=0.1)
    ax1.scatter(DRIFT_RANGE[min_mse_index], min_mse, alpha=0.1, c='k', marker='+')

In [8]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

example_set = 1

with open(f'{MAIN_FOLDER_PATH}/drift_{2}/simulation_set_{example_set}.pkl', 'rb') as file:
    simulations_set = pickle.load(file)

with open(f'{MAIN_FOLDER_PATH}/drift_{2}/hmm_fit_output_{example_set}.pkl', 'rb') as file:
    fit_output = pickle.load(file)

model = fit_output['models'][0]
choices_sequences_list = [synth_data['choices'] for synth_data in simulations_set]

average_proba_sequence_hmm = compute_reconstructed_average_proba_sequence(choices_sequences_list, model)

temp_mse_list = []

for test_average_probability_sequence in test_average_probability_sequences_list:
            
    temp_mse_list.append(compute_mean_square_error_v2(average_proba_sequence_hmm, test_average_probability_sequence))

min_mse = np.min(temp_mse_list)
min_mse_index = np.where(temp_mse_list==min_mse)[0]

ax1.plot(DRIFT_RANGE, temp_mse_list, alpha=1)
ax1.axvline(DRIFT_RANGE[min_mse_index], c='k', ls='--')
ax1.axvline(DRIFT_VALUES_ARR[2], color='k')

ax1.set_xlabel('Drift')
ax1.set_ylabel('MSE')


Text(0, 0.5, 'MSE')

In [ ]:
stacked_recovered_drift_per_states_number = []

for n in range(len(DRIFT_VALUES_ARR)):

    recovered_drift_per_states_number = []
    
    for n_states_index in range(len(STATES_NUMBER_RANGE)):

        recovered_drift_per_states_number.append([])

        for i in range(N_SETS):

            average_proba_sequence_hmm = []

            ##############################
            ### Loading Data and Model ###
            ##############################

            with open(f'{MAIN_FOLDER_PATH}/drift_{n}/simulation_set_{i}.pkl', 'rb') as file:
                simulations_set = pickle.load(file)

            with open(f'{MAIN_FOLDER_PATH}/drift_{n}/hmm_fit_output_{i}.pkl', 'rb') as file:
                fit_output = pickle.load(file)

            # model = fit_output['models'][np.argmax(fit_output['scores'])]
            model = fit_output['models'][n_states_index]

            ####################
            ### Computations ###
            ####################

            test_data = [simulation['choices'] for simulation in simulations_set]
            recovered_drift, _ = estimate_drift(simulations_set, model, test_average_probability_sequences_list, DRIFT_RANGE)
            
            recovered_drift_per_states_number[-1].append(recovered_drift)
            
    stacked_recovered_drift_per_states_number.append(recovered_drift_per_states_number)


In [14]:
example_drift = 3

fig=plt.figure(figsize=(4, 2.5), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])



for i in range(len(STATES_NUMBER_RANGE)):

    temp_recovered_drift_list = stacked_recovered_drift_per_states_number[example_drift][i]
    ax1.scatter(np.ones(N_SETS)*STATES_NUMBER_RANGE[i], temp_recovered_drift_list, marker='+', alpha=0.2, s=8)
    mean = np.mean(temp_recovered_drift_list)
    std = np.std(temp_recovered_drift_list)
    ax1.errorbar(STATES_NUMBER_RANGE[i],mean,std, marker='o', markersize=2, capsize=5)

ax1.axhline(DRIFT_VALUES_ARR[example_drift], color='k', linestyle='--')

# ax1.set_xticks([])
ax1.set_title(f'{N_SETS} sets of {SIMULATIONS_SET_SIZE} simulations')

ax1.set_ylim([0,0.12])
ax1.set_xlabel('Number of states in HMM')
ax1.set_ylabel('Recovered drift')

ax1.legend()

/tmp/ipykernel_6384/3153317469.py:28: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax1.legend()


In [ ]:
len(np.ones(SIMULATIONS_SET_SIZE)*STATES_NUMBER_RANGE[i])
len(temp_recovered_drift_list)

100

In [ ]:
DRIFT_VALUES_ARR

array([0.005  , 0.02875, 0.0525 , 0.07625, 0.1    ])

In [ ]:
# fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
# gs = fig.add_gridspec(1, 1)
# row = gs[:].subgridspec(1, 1, hspace=0.5)

# ax1 = plt.subplot(row[0,0])

# histo = np.histogram(recovered_drift_perdrift_list[4], bins=np.linspace(0.01,0.2,101))
# bin_width = histo[1][1] - histo[1][0]
# ax1.stairs(histo[0], histo[1], alpha=0.5, fill=True, label = 'Recovered drift probability density')

# ax1.axvline(0.050, linewidth=0.7, color='k', linestyle='--', label='Drift to recover')

# # ax1.set_xticks([])
# ax1.set_title(f'{N_SETS} sets of {SIMULATIONS_SET_SIZE} simulations')

# ax1.set_ylim([0,None])
# ax1.set_xlabel('Recovered drift')
# ax1.set_ylabel('Probability density')

# ax1.legend()